# Chain Rule & Automatic Differentiation Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: The Value class

In [ ]:
```python

class Value:

    def __init__(self, data, children=(), op=''):

        self.data = data

        self.grad = 0.0

        self._backward = lambda: None

        self._prev = set(children)

        self._op = op

    def __repr__(self):

        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

In [ ]:
```

Every `Value` stores its numeric data, its gradient (initially zero), a backward function, and pointers to child nodes that produced it.

### Step 2: Arithmetic operations with gradient tracking

In [ ]:
```python

    def __add__(self, other):

        other = other if isinstance(other, Value) else Value(other)

        out = Value(self.data + other.data, (self, other), '+')

        def _backward():

            self.grad += out.grad

            other.grad += out.grad

        out._backward = _backward

        return out

    def __mul__(self, other):

        other = other if isinstance(other, Value) else Value(other)

        out = Value(self.data * other.data, (self, other), '*')

        def _backward():

            self.grad += other.data * out.grad

            other.grad += self.data * out.grad

        out._backward = _backward

        return out

    def relu(self):

        out = Value(max(0, self.data), (self,), 'relu')

        def _backward():

            self.grad += (1.0 if out.data > 0 else 0.0) * out.grad

        out._backward = _backward

        return out

In [ ]:
```

Each operation creates a closure that knows how to compute local gradients and multiply by the upstream gradient (`out.grad`). The `+=` handles the case where a value is used in multiple operations.

### Step 3: The backward pass

In [ ]:
```python

    def backward(self):

        topo = []

        visited = set()

        def build_topo(v):

            if v not in visited:

                visited.add(v)

                for child in v._prev:

                    build_topo(child)

                topo.append(v)

        build_topo(self)

        self.grad = 1.0

        for v in reversed(topo):

            v._backward()

In [ ]:
```

Topological sort ensures every node's gradient is fully computed before it propagates to its children. The seed gradient is 1.0 (dy/dy = 1).

### Step 4: More operations for a complete engine

The basic Value class handles addition, multiplication, and relu. A real autograd engine needs more. Here are the operations you need to build neural networks:

In [ ]:
```python

    def __neg__(self):

        return self * -1

    def __sub__(self, other):

        return self + (-other)

    def __radd__(self, other):

        return self + other

    def __rmul__(self, other):

        return self * other

    def __rsub__(self, other):

        return other + (-self)

    def __pow__(self, n):

        out = Value(self.data ** n, (self,), f'**{n}')

        def _backward():

            self.grad += n * (self.data ** (n - 1)) * out.grad

        out._backward = _backward

        return out

    def __truediv__(self, other):

        return self * (other ** -1) if isinstance(other, Value) else self * (Value(other) ** -1)

    def exp(self):

        import math

        e = math.exp(self.data)

        out = Value(e, (self,), 'exp')

        def _backward():

            self.grad += e * out.grad

        out._backward = _backward

        return out

    def log(self):

        import math

        out = Value(math.log(self.data), (self,), 'log')

        def _backward():

            self.grad += (1.0 / self.data) * out.grad

        out._backward = _backward

        return out

    def tanh(self):

        import math

        t = math.tanh(self.data)

        out = Value(t, (self,), 'tanh')

        def _backward():

            self.grad += (1 - t ** 2) * out.grad

        out._backward = _backward

        return out

In [ ]:
```

**Why each operation matters:**

| Operation | Backward rule | Used in |

|-----------|--------------|---------|

| `__sub__` | Reuses add + neg | Loss computation (pred - target) |

| `__pow__` | n * x^(n-1) | Polynomial activations, MSE (error^2) |

| `__truediv__` | Reuses mul + pow(-1) | Normalization, learning rate scaling |

| `exp` | exp(x) * upstream | Softmax, log-likelihood |

| `log` | (1/x) * upstream | Cross-entropy loss, log probabilities |

| `tanh` | (1 - tanh^2) * upstream | Classic activation function |

The clever part: `__sub__` and `__truediv__` are defined in terms of existing operations. They get correct gradients for free because the chain rule composes through the underlying add/mul/pow operations.

### Step 5: Mini MLP from scratch

With a complete Value class, you can build a neural network. No PyTorch. No NumPy. Just Values and the chain rule.

In [ ]:
```python

import random

class Neuron:

    def __init__(self, n_inputs):

        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_inputs)]

        self.b = Value(0.0)

    def __call__(self, x):

        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)

        return act.tanh()

    def parameters(self):

        return self.w + [self.b]

class Layer:

    def __init__(self, n_inputs, n_outputs):

        self.neurons = [Neuron(n_inputs) for _ in range(n_outputs)]

    def __call__(self, x):

        return [n(x) for n in self.neurons]

    def parameters(self):

        return [p for n in self.neurons for p in n.parameters()]

class MLP:

    def __init__(self, sizes):

        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]

    def __call__(self, x):

        for layer in self.layers:

            x = layer(x)

        return x[0] if len(x) == 1 else x

    def parameters(self):

        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
```

A `Neuron` computes `tanh(w1*x1 + w2*x2 + ... + b)`. A `Layer` is a list of neurons. An `MLP` stacks layers. Every weight is a `Value`, so calling `loss.backward()` propagates gradients to every parameter.

**Training on XOR:**

In [ ]:
```python

random.seed(42)

model = MLP([2, 4, 1])  # 2 inputs, 4 hidden neurons, 1 output

xs = [[0, 0], [0, 1], [1, 0], [1, 1]]

ys = [-1, 1, 1, -1]  # XOR pattern (using -1/1 for tanh)

for step in range(100):

    preds = [model(x) for x in xs]

    loss = sum((p - y) ** 2 for p, y in zip(preds, ys))

    for p in model.parameters():

        p.grad = 0.0

    loss.backward()

    lr = 0.05

    for p in model.parameters():

        p.data -= lr * p.grad

    if step % 20 == 0:

        print(f"step {step:3d}  loss = {loss.data:.4f}")

print("\nPredictions after training:")

for x, y in zip(xs, ys):

    print(f"  input={x}  target={y:2d}  pred={model(x).data:6.3f}")

In [ ]:
```

This is micrograd. A complete neural network training loop in pure Python with automatic differentiation. Every commercial deep learning framework does the same thing at massive scale.

### Step 6: Gradient checking

How do you know your autodiff is correct? Compare it against numerical derivatives. This is gradient checking.

In [ ]:
```python

def gradient_check(build_expr, x_val, h=1e-7):

    x = Value(x_val)

    y = build_expr(x)

    y.backward()

    autodiff_grad = x.grad

    y_plus = build_expr(Value(x_val + h)).data

    y_minus = build_expr(Value(x_val - h)).data

    numerical_grad = (y_plus - y_minus) / (2 * h)

    diff = abs(autodiff_grad - numerical_grad)

    return autodiff_grad, numerical_grad, diff

In [ ]:
```

Test it on a complex expression:

In [ ]:
```python

def expr(x):

    return (x ** 3 + x * 2 + 1).tanh()

ad, num, diff = gradient_check(expr, 0.5)

print(f"Autodiff:  {ad:.8f}")

print(f"Numerical: {num:.8f}")

print(f"Difference: {diff:.2e}")

# Difference should be < 1e-5

In [ ]:
```

Gradient checking is essential when implementing new operations. If your backward pass has a bug, the numerical check catches it. Every serious deep learning implementation runs gradient checks during development.

**When to use gradient checking:**

| Situation | Do gradient check? |

|-----------|-------------------|

| Adding a new operation to your autograd | Yes, always |

| Debugging a training loop that won't converge | Yes, check gradients first |

| Production training | No, too slow (2x forward passes per parameter) |

| Unit tests for autograd code | Yes, automate it |

### Step 7: Verify against manual calculation

In [ ]:
```python

x1 = Value(2.0)

x2 = Value(3.0)

a = x1 * x2          # a = 6.0

b = a + Value(1.0)    # b = 7.0

y = b.relu()          # y = 7.0

y.backward()

print(f"y = {y.data}")          # 7.0

print(f"dy/dx1 = {x1.grad}")   # 3.0 (= x2)

print(f"dy/dx2 = {x2.grad}")   # 2.0 (= x1)

In [ ]:
```

Manual check: `y = relu(x1*x2 + 1)`. Since `x1*x2 + 1 = 7 > 0`, relu is identity.

`dy/dx1 = x2 = 3`. `dy/dx2 = x1 = 2`. The engine matches.

## Exercises

In [ ]:
1. Add `__pow__` to the Value class so you can compute `x ** n`. Verify that `d/dx(x^3)` at `x=2` equals `12.0`.

2. Add `tanh` as an activation function. Verify that `tanh'(0) = 1` and `tanh'(2) = 0.0707` (approx).

3. Build a computation graph for a single neuron: `y = relu(w1*x1 + w2*x2 + b)`. Compute all five gradients and verify against PyTorch.

4. Implement forward-mode autodiff using dual numbers. Create a `Dual` class and verify it gives the same derivatives as your reverse-mode engine.